# SPATIAL INTELLIGENCE — MULTI-FLOOR BUILDING (Marsella / Unité d'Habitation)

Analyze **three stacked floor plans** of Le Corbusier's *Unité d'Habitation* (Marseille) as a single connected building and run the Spatial Intelligence workflows (Session 03 / Assignment 02) **across all floors simultaneously**.

**Geometry note.** The source `Marsella_3-Floor-Plans.obj` contains three 2D plans. After importing with `Topology.ByOBJPath` (the project's Z-up convention) the three plans lie flat in the **XY plane** at three discrete levels **Z = 0, 4, 8**. Each plan is a long slab; its walls and stair shafts are the *holes* (the *Therme Vals* approach suggested by the instructor).

**Method (instructor's grid-sampling strategy).** For every floor we overlay a regular grid and keep the grid points that fall inside the navigable (meshed) area. Those points become graph nodes; neighbouring valid points are joined by edges. The three floor graphs are then **stacked** and **stitched together through stair nodes**, so that connectivity metrics such as **Degree Centrality are computed over the WHOLE building, not floor-by-floor.**

**Visual format.** Results are exported the same way as the S03 notebooks: each grid node becomes a small square cell coloured by the metric and rendered with `Topology.Show(faces, faceColorKey=...)` on a black background (filled heatmaps, not scatter points). The three floors are laid out one above the other in each heatmap.

Workflows included: Shortest Path (cross-floor), Closeness Centrality (Integration), Betweenness Centrality (Choice), Community Detection, building-wide Degree Centrality and per-floor Visibility/Isovist analysis.

## 1. Import the needed libraries

In [ ]:
import os, math, time
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Render every figure as a STATIC image via kaleido (no WebGL). This is what stops
# VS Code's "WebGL is not supported" crash: the interactive webview is never used.
pio.renderers.default = "png"

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Configuration

In [ ]:
renderer = "png"      # "png" renders via kaleido (no WebGL) — works in VS Code without GPU
                      # Change to "browser" or "notebook" if you want interactive figures

# Renderer used ONLY for the two 3D-graph cells you want to rotate/pan/zoom
# (building graph + shortest path). "browser" opens them in your default web browser,
# which is the most reliable way around VS Code's WebGL crash/freeze. Switch to
# "notebook" to embed them inline if your VS Code build handles WebGL well.
INTERACTIVE_RENDERER = "png"

# Paths
BASE_DIR  = r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Final_Project"
OBJ_PATH  = os.path.join(BASE_DIR, "3D-Models", "Marsella_3-Floor-Plans.obj")
ASSETS_DIR = os.path.join(BASE_DIR, "Notebooks", "assets", "assets_normal_building_floorplans_1.0m-grid")
os.makedirs(ASSETS_DIR, exist_ok=True)

# Floor levels (Z value of each plan after import) ordered bottom -> top
FLOOR_LEVELS = [0, 4, 8]
FLOOR_NAMES  = ["Floor 1", "Floor 2", "Floor 3"]

# Analysis grid spacing (plan units). Smaller = finer & slower. 3.0 is a good balance.
GRID_SIZE = 1


# Vertical spacing used to STACK the floors in the combined 3D graph (visual only)
FLOOR_HEIGHT = 30

# ---------------------------------------------------------------------------
# Stair / vertical-circulation locations in PLAN coordinates (X, Y).
# Floor indices: 0 = Floor 1 (Z=0), 1 = Floor 2 (Z=4), 2 = Floor 3 (Z=8).
# ---------------------------------------------------------------------------

# Floor 3 <-> Floor 2  (pair (1, 2))   [24 points]
LEVEL_3_TO_2 = [
    (337.5, 22), (335.5, 22), (327.5, 22), (329.5, 22),
    (321.5, 22), (319.5, 22), (311.5, 22), (313.5, 22),
    (265.5, 22), (263.5, 22), (271.5, 22), (273.5, 22),
    (281.5, 22), (279.5, 22), (287.5, 22), (289.5, 22),
    (257.5, 22), (255.5, 22), (241.5, 22), (239.5, 22),
    (231.5, 22), (233.5, 22), (295.5, 22.0), (249.5, 22.0),
]

# Floor 1 <-> Floor 2  (pair (0, 1))   [31 points]
LEVEL_2_TO_1 = [
    (311.5, 12), (313.5, 12), (319.5, 12), (321.5, 12),
    (335.5, 12), (337.5, 12), (327.5, 12), (329.5, 12),
    (247.5, 12), (249.5, 12), (287.5, 12), (289.5, 12),
    (279.5, 12), (281.5, 12), (271.5, 12), (273.5, 12),
    (255.5, 12), (257.5, 12), (263.5, 12), (265.5, 12),
    (239.5, 12), (241.5, 12), (231.5, 12), (233.5, 12),
    (354.5, 22), (354.5, 16), (354.5, 10), (343.5, 12),
    (295.5, 12), (297.5, 12), (305.5, 12),
]

# All three floors  (pairs (0, 1) and (1, 2))   [3 points]
LEVEL_ALL = [
    (341.5, 24.0), (299.5, 24.0), (247.5, 24.0),
]

STAIR_LOCATIONS = (
    [(x, y, [(1, 2)])         for (x, y) in LEVEL_3_TO_2]
    + [(x, y, [(0, 1)])         for (x, y) in LEVEL_2_TO_1]
    + [(x, y, [(0, 1), (1, 2)]) for (x, y) in LEVEL_ALL]
)
print(f"{len(STAIR_LOCATIONS)} stair locations "
      f"({len(LEVEL_3_TO_2)} for 3<->2, {len(LEVEL_2_TO_1)} for 2<->1, "
      f"{len(LEVEL_ALL)} for all)")

SAVE_IMAGES = True

def save_fig(fig, filename):
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1800, height=1100, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Utility functions

* `extract_triangles` / `points_inside` — geometry helpers (pull triangles, point-in-mesh test).
* `find_closest_node` — the instructor's `find_closest_vertex`, snaps a stair location to the nearest grid node.
* `make_cell_face` + `show_face_heatmap` — turn each grid node into a filled square cell and render it as a `Topology.Show` heatmap (S03 export format).

In [ ]:
def extract_triangles(face_list):
    # Return (T,3,2) array of plan-space (X,Y) triangles for a list of topologic faces.
    tris = []
    for f in face_list:
        vs = Topology.Vertices(f)
        pts = [(Vertex.X(v), Vertex.Y(v)) for v in vs]
        for i in range(1, len(pts) - 1):          # fan-triangulate (faces are already triangles)
            tris.append([pts[0], pts[i], pts[i + 1]])
    return np.array(tris)

def points_inside(tris, P):
    # Boolean mask: which points in P (N,2) fall inside ANY triangle of tris (T,3,2).
    a, b, c = tris[:, 0], tris[:, 1], tris[:, 2]
    v0 = b - a; v1 = c - a
    d00 = (v0 * v0).sum(1); d01 = (v0 * v1).sum(1); d11 = (v1 * v1).sum(1)
    den = d00 * d11 - d01 * d01
    den[den == 0] = 1e-12
    inside = np.zeros(len(P), bool)
    for i, p in enumerate(P):
        v2 = p - a
        d20 = (v2 * v0).sum(1); d21 = (v2 * v1).sum(1)
        u = (d11 * d20 - d01 * d21) / den
        w = (d00 * d21 - d01 * d20) / den
        if np.any((u >= -1e-6) & (w >= -1e-6) & (u + w <= 1 + 1e-6)):
            inside[i] = True
    return inside

def find_closest_node(node_xy, x, y):
    # Index of the grid node closest to (x, y) -- the instructor's find_closest_vertex.
    d = (node_xy[:, 0] - x) ** 2 + (node_xy[:, 1] - y) ** 2
    return int(d.argmin())

def rk(u, v):
    return (round(float(u), 3), round(float(v), 3))

def make_cell_face(cx, cy, h):
    # A flat square cell of half-size h centred at (cx, cy), z = 0.
    pts = [Vertex.ByCoordinates(cx - h, cy - h, 0.0), Vertex.ByCoordinates(cx + h, cy - h, 0.0),
           Vertex.ByCoordinates(cx + h, cy + h, 0.0), Vertex.ByCoordinates(cx - h, cy + h, 0.0)]
    return Face.ByWire(Wire.ByVertices(pts, close=True))

def show_face_heatmap(faces_values, title, filename, colorScale="viridis"):
    # faces_values: list of (face, value). Colours each cell and renders the filled
    # heatmap with Topology.Show, exactly like the S03 notebooks (faceColorKey, black bg).
    vals = [v for _, v in faces_values]
    mn, mx = float(min(vals)), float(max(vals))
    if mx == mn: mx = mn + 1e-9
    for f, val in faces_values:
        col = Color.AnyToHex(Color.ByValueInRange(float(val), minValue=mn, maxValue=mx, colorScale=colorScale))
        d = Topology.Dictionary(f)
        d = Dictionary.SetValueAtKey(d, "hm_color", col)
        Topology.SetDictionary(f, d)
    faces = [f for f, _ in faces_values]
    fig = Topology.Show(faces, faceColorKey="hm_color", faceOpacity=1.0,
                        showEdges=False, showVertices=False, camera=[0, 0, 6],
                        backgroundColor="black", width=1700, height=1000,
                        showFigure=False, renderer=renderer)
    for fi in range(len(FLOOR_LEVELS)):
        yc = (fi - 1) * ROWGAP
        fig.add_trace(go.Scatter3d(x=[-(UMAX - UMIN) / 2 - 7], y=[yc], z=[0], mode="text",
                                   text=[FLOOR_NAMES[fi]], textfont=dict(color="white", size=16),
                                   showlegend=False))
    fig.update_layout(title=dict(text=title, font=dict(color="white")))
    fig.show(renderer=renderer)
    save_fig(fig, filename)
    return fig

## 5. Import the OBJ and split it into the three floor plans

`Topology.ByOBJPath` returns clusters of triangulated faces. We collect every face and bin it by its centroid's Z value into the three floor levels, then compute the shared plan bounding box.

In [ ]:
result = Topology.ByOBJPath(OBJ_PATH)
all_faces = []
for item in result:
    if Topology.IsInstance(item, "Cluster"):
        cf = Cluster.Faces(item)
        if cf: all_faces.extend(cf)
    elif Topology.IsInstance(item, "Face"):
        all_faces.append(item)
print(f"Imported {len(all_faces)} triangulated faces")

floor_faces = {lv: [] for lv in FLOOR_LEVELS}
for f in all_faces:
    z = Vertex.Z(Topology.Centroid(f))
    lv = min(FLOOR_LEVELS, key=lambda k: abs(k - z))
    floor_faces[lv].append(f)
for i, lv in enumerate(FLOOR_LEVELS):
    print(f"  {FLOOR_NAMES[i]} (Z={lv}): {len(floor_faces[lv])} faces")

# Shared plan bounding box + vertical row gap used to stack floors in the 2D heatmaps
allxy = np.vstack([extract_triangles(floor_faces[lv]).reshape(-1, 2) for lv in FLOOR_LEVELS])
UMIN, VMIN = allxy.min(0)
UMAX, VMAX = allxy.max(0)
UMID, VMID = 0.5 * (UMIN + UMAX), 0.5 * (VMIN + VMAX)
ROWGAP = (VMAX - VMIN) + 8.0
print(f"Plan box: u[{UMIN:.1f},{UMAX:.1f}]  v[{VMIN:.1f},{VMAX:.1f}]")

## 6. Show the three raw floor plans

In [ ]:
fig = make_subplots(rows=len(FLOOR_LEVELS), cols=1,
                    subplot_titles=[f"{FLOOR_NAMES[i]} (Z={FLOOR_LEVELS[i]})" for i in range(len(FLOOR_LEVELS))])
for i, lv in enumerate(FLOOR_LEVELS):
    tris = extract_triangles(floor_faces[lv])
    for t in tris:
        xs = list(t[:, 0]) + [t[0, 0]]
        ys = list(t[:, 1]) + [t[0, 1]]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", fill="toself",
                                 line=dict(color="rgba(170,195,255,0.45)", width=0.4),
                                 fillcolor="rgba(70,120,235,0.35)", showlegend=False), row=i + 1, col=1)
    fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=i + 1, col=1)
fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
for ann in fig.layout.annotations:
    ann.font.color = "white"
fig.update_layout(title=dict(text="Three imported floor plans (plan view)", font=dict(color="white")),
                  height=300 * len(FLOOR_LEVELS), width=1500,
                  paper_bgcolor="black", plot_bgcolor="black")
fig.show(renderer=renderer)
save_fig(fig, "01_floor_plans.png")

## 7. Sample a navigable grid on each floor

A regular grid is laid over the shared bounding box; only points inside the meshed (navigable) area are kept. These become the graph nodes of each floor.

In [ ]:
us = np.arange(UMIN, UMAX + GRID_SIZE, GRID_SIZE)
vs = np.arange(VMIN, VMAX + GRID_SIZE, GRID_SIZE)
UU, VV = np.meshgrid(us, vs)
GRID_PTS = np.column_stack([UU.ravel(), VV.ravel()])

floor_valid = {}    # level -> valid_xy ndarray
for i, lv in enumerate(FLOOR_LEVELS):
    tris = extract_triangles(floor_faces[lv])
    floor_valid[lv] = GRID_PTS[points_inside(tris, GRID_PTS)]
    print(f"  {FLOOR_NAMES[i]} (Z={lv}): {len(floor_valid[lv])} navigable nodes")

## 8. Show the navigable grids (use these coordinates to locate your stairs)

In [ ]:
fig = make_subplots(rows=len(FLOOR_LEVELS), cols=1,
                    subplot_titles=[f"{FLOOR_NAMES[i]} (Z={FLOOR_LEVELS[i]})" for i in range(len(FLOOR_LEVELS))])
for i, lv in enumerate(FLOOR_LEVELS):
    valid = floor_valid[lv]
    fig.add_trace(go.Scatter(x=valid[:, 0], y=valid[:, 1], mode="markers",
                             marker=dict(size=5, color="royalblue"), showlegend=False), row=i + 1, col=1)
    sx = [s[0] for s in STAIR_LOCATIONS]; sy = [s[1] for s in STAIR_LOCATIONS]
    fig.add_trace(go.Scatter(x=sx, y=sy, mode="markers",
                             marker=dict(size=14, color="red", symbol="x"), name="stairs",
                             showlegend=(i == 0)), row=i + 1, col=1)
    fig.update_yaxes(scaleanchor=f"x{i+1 if i else ''}", scaleratio=1, row=i + 1, col=1)
fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.10)", color="white", zeroline=False)
for ann in fig.layout.annotations:
    ann.font.color = "white"
fig.update_layout(title=dict(text="Navigable grids + stair locations (red x)", font=dict(color="white")),
                  height=300 * len(FLOOR_LEVELS), width=1500,
                  paper_bgcolor="black", plot_bgcolor="black",
                  legend=dict(font=dict(color="white")))
fig.show(renderer=renderer)
save_fig(fig, "02_navigable_grids.png")

## 9. Build the per-floor graphs, the display cells, and stack everything

For each floor the valid points become topologic vertices at `Z = floor_index * FLOOR_HEIGHT` (for the 3D graph) and a flat square **display cell** centred at `(u, v + offset)` (for the 2D heatmaps). Horizontal edges join 4-neighbour valid points.

In [ ]:
all_v = []            # topologic vertices (all floors, stacked in Z)
all_e = []            # topologic edges
floor_index_map = {}  # level -> {(round u, round v): global vertex index}
display_faces = []    # flat square cells for the heatmaps
cell_lookup = {}      # (floor_index, (round u, round v)) -> display face
H = GRID_SIZE / 2.0

for fi, lv in enumerate(FLOOR_LEVELS):
    valid = floor_valid[lv]
    z = fi * FLOOR_HEIGHT
    yoff = (fi - 1) * ROWGAP
    idx = {}
    for (u, v) in valid:
        idx[rk(u, v)] = len(all_v)
        all_v.append(Vertex.ByCoordinates(float(u), float(v), float(z)))
        cell = make_cell_face(float(u) - UMID, (float(v) - VMID) + yoff, H)
        display_faces.append(cell)
        cell_lookup[(fi, rk(u, v))] = cell
    floor_index_map[lv] = idx
    ne = 0
    for (u, v) in valid:
        for du, dv in [(GRID_SIZE, 0), (0, GRID_SIZE)]:
            k = rk(u + du, v + dv)
            if k in idx:
                all_e.append(Edge.ByVertices([all_v[idx[rk(u, v)]], all_v[idx[k]]]))
                ne += 1
    print(f"  {FLOOR_NAMES[fi]}: {len(valid)} nodes, {ne} horizontal edges")
print(f"Subtotal: {len(all_v)} nodes, {len(all_e)} horizontal edges, {len(display_faces)} display cells")

def heatmap_from_graph(values, title, filename, colorScale):
    # Pair every graph vertex with its display cell, then render the filled heatmap.
    fv = []
    for v, val in zip(gverts, values):
        fi = int(round(Vertex.Z(v) / FLOOR_HEIGHT))
        f = cell_lookup.get((fi, rk(Vertex.X(v), Vertex.Y(v))))
        if f is not None:
            fv.append((f, val))
    return show_face_heatmap(fv, title, filename, colorScale)

## 10. Connect the floors through the stairs

For every stair location and adjacent floor pair we snap to the closest navigable node on each floor (`find_closest_node`) and add a **vertical stair edge** — this turns three separate plans into one connected building.

In [ ]:
stair_node_pairs = []
for stair in STAIR_LOCATIONS:
    sx, sy = stair[0], stair[1]
    # Use the floor pairs declared for this stair; otherwise connect every adjacent floor.
    pairs = stair[2] if len(stair) >= 3 and stair[2] else [(i, i + 1) for i in range(len(FLOOR_LEVELS) - 1)]
    for fa, fb in pairs:
        a, b = FLOOR_LEVELS[fa], FLOOR_LEVELS[fb]
        va, vb = floor_valid[a], floor_valid[b]
        ia = find_closest_node(va, sx, sy)
        ib = find_closest_node(vb, sx, sy)
        gia = floor_index_map[a][rk(va[ia, 0], va[ia, 1])]
        gib = floor_index_map[b][rk(vb[ib, 0], vb[ib, 1])]
        all_e.append(Edge.ByVertices([all_v[gia], all_v[gib]]))
        stair_node_pairs.append((gia, gib))
print(f"Added {len(stair_node_pairs)} vertical stair edges "
      f"from {len(STAIR_LOCATIONS)} stair location(s)")

## 11. Build the combined BUILDING graph

In [ ]:
t0 = time.time()
building_graph = Graph.ByVerticesEdges(all_v, all_e)
gverts = Graph.Vertices(building_graph)
gedges = Graph.Edges(building_graph)
print(f"Building graph: {len(gverts)} vertices, {len(gedges)} edges  ({time.time()-t0:.1f}s)")
print(f"Graph density:  {Graph.Density(building_graph):.5f}")

## 12. Show the combined 3D building graph

Three stacked floors connected by the vertical stair edges (highlighted in red).

In [ ]:
fig = Topology.Show(building_graph,
                   vertexSize=3, vertexColor="royalblue",
                   edgeColor="lightgrey", edgeWidth=1,
                   backgroundColor="black",
                   width=1400, height=900,
                   showFigure=False, renderer=renderer)
for (a, b) in stair_node_pairs:
    pa, pb = all_v[a], all_v[b]
    fig.add_trace(go.Scatter3d(x=[Vertex.X(pa), Vertex.X(pb)], y=[Vertex.Y(pa), Vertex.Y(pb)],
                               z=[Vertex.Z(pa), Vertex.Z(pb)], mode="lines",
                               line=dict(color="red", width=6), showlegend=False))
fig.update_layout(
    scene=dict(aspectmode="data",
               xaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="x", font=dict(color="white"))),
               yaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="y", font=dict(color="white"))),
               zaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="z (floor stack)", font=dict(color="white")))),
    # Zoom out so all three stacked floors fit in the static PNG export
    scene_camera=dict(projection=dict(type="orthographic"), eye=dict(x=1.4, y=1.4, z=1.4)),
    scene_dragmode="orbit"
)
fig.show(renderer=INTERACTIVE_RENDERER)
save_fig(fig, "03_building_graph_3d.png")

## 13. Minimum Spanning Tree

The **Minimum Spanning Tree** (MST) connects all building nodes with the fewest edges and minimum total weight — the irreducible skeleton that keeps all spaces reachable. Comparing MST density against the full graph reveals how much **redundancy** (alternative paths / loops) the layout provides. A higher ratio of original-to-MST density means a more resilient, multiply-connected circulation network.

In [ ]:
t0 = time.time()
mst = Graph.MinimumSpanningTree(building_graph)
mst_verts = Graph.Vertices(mst)
mst_edges = Graph.Edges(mst)
print(f"Minimum Spanning Tree: {len(mst_verts)} vertices, {len(mst_edges)} edges  ({time.time()-t0:.1f}s)")
print(f"Original graph — Density: {Graph.Density(building_graph):.5f}")
print(f"MST            — Density: {Graph.Density(mst):.5f}")

edge_x, edge_y, edge_z = [], [], []
for e in mst_edges:
    vs_e = Topology.Vertices(e)
    if len(vs_e) >= 2:
        edge_x += [Vertex.X(vs_e[0]), Vertex.X(vs_e[1]), None]
        edge_y += [Vertex.Y(vs_e[0]), Vertex.Y(vs_e[1]), None]
        edge_z += [Vertex.Z(vs_e[0]), Vertex.Z(vs_e[1]), None]

vx = [Vertex.X(v) for v in mst_verts]
vy = [Vertex.Y(v) for v in mst_verts]
vz = [Vertex.Z(v) for v in mst_verts]

_axx = dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="x", font=dict(color="white")))
_axy = dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="y", font=dict(color="white")))
_axz = dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="z (floor stack)", font=dict(color="white")))

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode="lines",
                           line=dict(color="lightgrey", width=1), showlegend=False))
fig.add_trace(go.Scatter3d(x=vx, y=vy, z=vz, mode="markers",
                           marker=dict(size=2, color="royalblue"), showlegend=False))
fig.update_layout(
    title=dict(text="Minimum Spanning Tree — whole building", font=dict(color="white")),
    paper_bgcolor="black",
    scene=dict(aspectmode="data",
               xaxis=_axx, yaxis=_axy, zaxis=_axz),
    # Zoom out so all floors fit in the static PNG export
    scene_camera=dict(projection=dict(type="orthographic"), eye=dict(x=1.4, y=1.4, z=1.4)),
    scene_dragmode="orbit",
    width=1400, height=900
)
fig.show(renderer=INTERACTIVE_RENDERER)
save_fig(fig, "10_mst_3d.png")

## 14. Cross-floor Shortest Path (+ Straightened)

Navigate from one end of the **bottom** floor to the far end of the **top** floor. The route climbs through stair nodes, demonstrating genuine cross-floor connectivity. `Wire.Straighten` is attempted to reduce unnecessary turns and shorten the path.

In [ ]:
def graph_closest(x, y, z):
    best, bd = None, 1e18
    for v in gverts:
        d = (Vertex.X(v) - x)**2 + (Vertex.Y(v) - y)**2 + (Vertex.Z(v) - z)**2
        if d < bd: bd, best = d, v
    return best

start_v = graph_closest(UMIN + GRID_SIZE, VMID, 0)
end_v   = graph_closest(UMAX - GRID_SIZE, VMID, (len(FLOOR_LEVELS) - 1) * FLOOR_HEIGHT)

t0 = time.time()
path = Graph.ShortestPath(building_graph, vertexA=start_v, vertexB=end_v)
key_verts = None
simp_len  = 0.0

if path is None:
    print("No path found — check STAIR_LOCATIONS.")
else:
    path_len     = Wire.Length(path)
    path_verts_p = Topology.Vertices(path)
    path_edges_p = Topology.Edges(path)
    n_turns      = max(0, len(path_verts_p) - 2)
    print(f"Cross-floor shortest path  ({time.time()-t0:.1f}s)")
    print(f"  Length : {path_len:.1f}")
    print(f"  Edges  : {len(path_edges_p)}")
    print(f"  Turns  : {n_turns}")

    # Simplified path — keep only floor-transition (stair) nodes + start/end.
    # Equivalent to Wire.Straighten for a 3D cross-floor path: removes same-floor
    # zigzag, leaving just start → stair → stair → ... → end connected by straight
    # lines. Wire.Straighten requires a 2D host Face so it cannot run here directly.
    kv = [path_verts_p[0]]
    for i in range(1, len(path_verts_p) - 1):
        z_prev = Vertex.Z(path_verts_p[i - 1])
        z_curr = Vertex.Z(path_verts_p[i])
        z_next = Vertex.Z(path_verts_p[i + 1])
        if abs(z_curr - z_prev) > 0.1 or abs(z_curr - z_next) > 0.1:
            kv.append(path_verts_p[i])
    kv.append(path_verts_p[-1])
    seen, key_verts = set(), []
    for v in kv:
        k = (round(Vertex.X(v), 1), round(Vertex.Y(v), 1), round(Vertex.Z(v), 1))
        if k not in seen:
            seen.add(k); key_verts.append(v)

    simp_len = sum(
        math.sqrt((Vertex.X(key_verts[i+1]) - Vertex.X(key_verts[i]))**2 +
                  (Vertex.Y(key_verts[i+1]) - Vertex.Y(key_verts[i]))**2 +
                  (Vertex.Z(key_verts[i+1]) - Vertex.Z(key_verts[i]))**2)
        for i in range(len(key_verts) - 1))
    reduction = (path_len - simp_len) / path_len * 100
    print(f"\nSimplified path (stair waypoints only):")
    print(f"  Length    : {simp_len:.1f}  (−{reduction:.1f}% vs full path)")
    print(f"  Waypoints : {len(key_verts)}  (from {len(path_verts_p)} nodes)")

# ── Figure ────────────────────────────────────────────────────────────────────
fig = go.Figure()

# 1 — Ghosted floor surfaces: semi-transparent Mesh3d at each stacked Z level
for fi, lv in enumerate(FLOOR_LEVELS):
    tris    = extract_triangles(floor_faces[lv])
    z_level = fi * FLOOR_HEIGHT
    vx, vy, vz, ii, jj, kk = [], [], [], [], [], []
    for ti, t in enumerate(tris):
        base = ti * 3
        vx += [t[0, 0], t[1, 0], t[2, 0]]
        vy += [t[0, 1], t[1, 1], t[2, 1]]
        vz += [z_level, z_level, z_level]
        ii.append(base); jj.append(base + 1); kk.append(base + 2)
    fig.add_trace(go.Mesh3d(
        x=vx, y=vy, z=vz, i=ii, j=jj, k=kk,
        opacity=0.13, color="steelblue",
        showscale=False, showlegend=False, hoverinfo="skip",
        flatshading=True))

# 2 — Graph nodes (faint background, no edges drawn to keep scene readable)
fig.add_trace(go.Scatter3d(
    x=[Vertex.X(v) for v in gverts],
    y=[Vertex.Y(v) for v in gverts],
    z=[Vertex.Z(v) for v in gverts],
    mode="markers", marker=dict(size=1.5, color="rgba(160,180,220,0.3)"),
    showlegend=False, hoverinfo="skip"))

# 3 — Stair connections (semi-transparent orange-red)
for (a, b) in stair_node_pairs:
    pa, pb = all_v[a], all_v[b]
    fig.add_trace(go.Scatter3d(
        x=[Vertex.X(pa), Vertex.X(pb)],
        y=[Vertex.Y(pa), Vertex.Y(pb)],
        z=[Vertex.Z(pa), Vertex.Z(pb)],
        mode="lines", line=dict(color="rgba(255,100,50,0.4)", width=2),
        showlegend=False, hoverinfo="skip"))

# 4 — Shortest path — RED (full zigzag route through graph)
if path is not None:
    pv = Topology.Vertices(path)
    fig.add_trace(go.Scatter3d(
        x=[Vertex.X(v) for v in pv],
        y=[Vertex.Y(v) for v in pv],
        z=[Vertex.Z(v) for v in pv],
        mode="lines+markers",
        line=dict(color="red", width=7),
        marker=dict(size=3, color="red"),
        name=f"Shortest path  L={path_len:.0f}"))

# 5 — Simplified path — LIME/GREEN (stair waypoints connected straight)
if key_verts is not None:
    fig.add_trace(go.Scatter3d(
        x=[Vertex.X(v) for v in key_verts],
        y=[Vertex.Y(v) for v in key_verts],
        z=[Vertex.Z(v) for v in key_verts],
        mode="lines+markers",
        line=dict(color="#00ff88", width=6, dash="dot"),
        marker=dict(size=7, color="#00ff88", symbol="diamond"),
        name=f"Simplified  L={simp_len:.0f}  ({len(key_verts)} pts)"))

# 6 — Start / End markers
if path is not None:
    for v, c, lbl in [(start_v, "cyan", "Start"), (end_v, "orange", "End")]:
        fig.add_trace(go.Scatter3d(
            x=[Vertex.X(v)], y=[Vertex.Y(v)], z=[Vertex.Z(v)],
            mode="markers+text",
            marker=dict(size=13, color=c, symbol="diamond"),
            text=[lbl], textposition="top center",
            textfont=dict(color=c, size=13),
            showlegend=False))

fig.update_layout(
    paper_bgcolor="black",
    scene=dict(aspectmode="data",
               xaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="x", font=dict(color="white"))),
               yaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="y", font=dict(color="white"))),
               zaxis=dict(showbackground=False, showgrid=True, gridcolor="rgba(255,255,255,0.15)", zeroline=False, color="white", title=dict(text="z (floor stack)", font=dict(color="white")))),
    # Zoom out so all three floors + path fit in the static PNG export
    scene_camera=dict(projection=dict(type="orthographic"), eye=dict(x=1.4, y=1.4, z=1.4)),
    scene_dragmode="orbit",
    legend=dict(font=dict(color="white", size=13), bgcolor="rgba(0,0,0,0.65)",
                x=0.01, y=0.99),
    width=1400, height=900
)
fig.show(renderer=INTERACTIVE_RENDERER)
save_fig(fig, "08_shortest_path_3d.png")

## 15. Building-wide DEGREE CENTRALITY

> **Key building-scale metric.** Degree centrality is computed on the **whole-building graph** (all three floors connected through the stairs), so stair nodes and well-connected circulation spaces score across floors — not per plan in isolation. The heatmap below shows the three floors stacked bottom-to-top.

In [ ]:
t0 = time.time()
degree_values = Graph.DegreeCentrality(building_graph, normalize=True)
n_nodes = len(degree_values)
a = np.array(degree_values, dtype=float)
print(f"Degree centrality — {n_nodes} nodes  ({time.time()-t0:.1f}s)")
print(f"  Range : {a.min():.4f} – {a.max():.4f}")
print(f"  Mean  : {a.mean():.4f}   Std: {a.std():.4f}")

sorted_dc = sorted(zip(gverts, degree_values), key=lambda x: x[1], reverse=True)

print("\nTop 5 most connected (hubs):")
for v, score in sorted_dc[:5]:
    fl    = int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1
    links = round(score * (n_nodes - 1))
    print(f"  Floor {fl}  ({Vertex.X(v):.1f}, {Vertex.Y(v):.1f})  →  {score:.4f}  ({links} connections)")

print("\nTop 5 least connected (dead ends):")
for v, score in sorted_dc[-5:]:
    fl    = int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1
    links = round(score * (n_nodes - 1))
    print(f"  Floor {fl}  ({Vertex.X(v):.1f}, {Vertex.Y(v):.1f})  →  {score:.4f}  ({links} connections)")

heatmap_from_graph(degree_values, "Degree Centrality (whole building)",
                   "04_degree_centrality.png", "viridis")

## 16. Closeness Centrality (Integration)

How close each space is to every other space in the **entire building**. High values = globally integrated, easy-to-reach locations.

In [ ]:
t0 = time.time()
closeness_values = Graph.ClosenessCentrality(building_graph)
a = np.array(closeness_values, dtype=float)
print(f"Closeness centrality — {len(closeness_values)} nodes  ({time.time()-t0:.1f}s)")
print(f"  Range : {a.min():.4f} – {a.max():.4f}")
print(f"  Mean  : {a.mean():.4f}   Std: {a.std():.4f}")

sorted_cc = sorted(zip(gverts, closeness_values), key=lambda x: x[1], reverse=True)
print("\nTop 5 most integrated (closest to all):")
for v, score in sorted_cc[:5]:
    fl = int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1
    print(f"  Floor {fl}  ({Vertex.X(v):.1f}, {Vertex.Y(v):.1f})  →  {score:.4f}")

heatmap_from_graph(closeness_values, "Closeness Centrality / Integration (whole building)",
                   "05_closeness_centrality.png", "thermal")

## 17. Betweenness Centrality (Choice)

How often each space lies on the shortest paths between all other spaces. High values = critical circulation routes; the stair nodes light up because every cross-floor trip passes through them.

In [ ]:
t0 = time.time()
betweenness_values = Graph.BetweennessCentrality(building_graph, normalize=True)
a = np.array(betweenness_values, dtype=float)
print(f"Betweenness centrality — {len(betweenness_values)} nodes  ({time.time()-t0:.1f}s)")
print(f"  Range : {a.min():.4f} – {a.max():.4f}")
print(f"  Mean  : {a.mean():.4f}   Std: {a.std():.4f}")

sorted_bc = sorted(zip(gverts, betweenness_values), key=lambda x: x[1], reverse=True)
print("\nTop 5 most traversed spaces (critical paths):")
for v, score in sorted_bc[:5]:
    fl = int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1
    print(f"  Floor {fl}  ({Vertex.X(v):.1f}, {Vertex.Y(v):.1f})  →  {score:.4f}")

heatmap_from_graph(betweenness_values, "Betweenness Centrality / Choice (whole building)",
                   "06_betweenness_centrality.png", "thermal")

## 18. Community Detection

Partition the building into spatial communities — densely connected groups of nodes. Because the graph spans all floors, a community can extend vertically through a stair.

In [ ]:
from collections import Counter
t0 = time.time()
community_values = Graph.CommunityPartition(building_graph)
n_comm = len(set(community_values))
community_counts = Counter(community_values)
cell_area_m2 = GRID_SIZE ** 2

print(f"Detected {n_comm} communities  ({time.time()-t0:.1f}s)")
print(f"\n{'ID':>4}  {'Cells':>6}  {'Area m²':>9}")
for cid, cnt in sorted(community_counts.items()):
    print(f"  {cid:>3}    {cnt:>5}    {cnt * cell_area_m2:>7.2f}")

largest_id  = max(community_counts, key=community_counts.get)
smallest_id = min(community_counts, key=community_counts.get)
total_area  = len(community_values) * cell_area_m2
print(f"\nLargest:  Community {largest_id} — {community_counts[largest_id]} cells, "
      f"~{community_counts[largest_id] * cell_area_m2:.2f} m²")
print(f"Smallest: Community {smallest_id} — {community_counts[smallest_id]} cells, "
      f"~{community_counts[smallest_id] * cell_area_m2:.2f} m²")
print(f"Total navigable cells: {len(community_values)},  area ~{total_area:.2f} m²")

heatmap_from_graph(community_values, f"Community Detection — {n_comm} communities (whole building)",
                   "07_communities.png", "rainbow")

## 19. Visibility Heatmap / VGA (per floor)

Unlike the connectivity metrics, **visibility does not cross floors**, so this Visibility Graph Analysis is computed **independently per floor**. We scatter a coarse grid of viewpoints and connect every pair whose sightline stays inside the floor; the **visibility degree** = how many other viewpoints each one can see. Rendered as filled cells, same export format as the centrality heatmaps.

In [ ]:
VGA_GRID_SIZE = 5   # spacing of the isovist viewpoints (coarser than the analysis grid)
VIS_SAMPLES   = 12     # samples along each sightline for the occlusion test

def compute_visibility(tris, viewpoints):
    n = len(viewpoints)
    ts = np.linspace(0.12, 0.88, VIS_SAMPLES)
    deg = np.zeros(n, int)
    for i in range(n):
        for j in range(i + 1, n):
            seg = viewpoints[i][None, :] * (1 - ts)[:, None] + viewpoints[j][None, :] * ts[:, None]
            if points_inside(tris, seg).all():
                deg[i] += 1; deg[j] += 1
    return deg

uvp = np.arange(UMIN, UMAX + VGA_GRID_SIZE, VGA_GRID_SIZE)
vvp = np.arange(VMIN, VMAX + VGA_GRID_SIZE, VGA_GRID_SIZE)
UUv, VVv = np.meshgrid(uvp, vvp)
VP_PTS = np.column_stack([UUv.ravel(), VVv.ravel()])
Hv = VGA_GRID_SIZE / 2.0

vis_faces_values = []
for fi, lv in enumerate(FLOOR_LEVELS):
    tris = extract_triangles(floor_faces[lv])
    vp = VP_PTS[points_inside(tris, VP_PTS)]
    t0 = time.time()
    deg = compute_visibility(tris, vp)
    yoff = (fi - 1) * ROWGAP
    for (u, v), dval in zip(vp, deg):
        cell = make_cell_face(float(u) - UMID, (float(v) - VMID) + yoff, Hv)
        vis_faces_values.append((cell, int(dval)))
    print(f"  {FLOOR_NAMES[fi]} (Z={lv}): {len(vp)} viewpoints, "
          f"visibility {deg.min()}-{deg.max()} (mean {deg.mean():.1f}, {time.time()-t0:.1f}s)")

show_face_heatmap(vis_faces_values, "Visibility Graph Analysis - isovist degree (per floor)",
                  "09_visibility_isovist.png", "plasma")

## 20. Isovist Analysis (per floor)

An **isovist** is the exact polygon of space visible from a single viewpoint — the field of vision cast across the floor plan. Unlike the VGA heatmap (section 19) which counts mutual visibility between grid points, `Face.Isovist` computes the actual **geometric visibility shape** from each viewpoint.

**Method:** For each floor we reconstruct a clean planar `Face` (external boundary + internal wall holes) using `Shell.ByFaces`. If successful, `Face.Isovist` casts rays against the wall geometry and returns the visibility polygon. Floors where the OBJ triangulation is too dense for clean shell reconstruction are skipped — their visibility is already captured in section 19.

In [ ]:
from topologicpy.Shell import Shell as _Shell

# Attempt to reconstruct a clean planar Face per floor for Face.Isovist.
# Shell.ByFaces works only where OBJ triangulation is non-overlapping (typically Z=4).
# Floors that fail fall back gracefully — their VGA heatmap (section 19) covers them.
ISO_STEP = 4.0   # viewpoint spacing; Face.Isovist is slow, keep sparse

floor_face_clean = {}
for fi, lv in enumerate(FLOOR_LEVELS):
    try:
        s = _Shell.ByFaces(floor_faces[lv])
        if s is None:
            print(f"  {FLOOR_NAMES[fi]}: Shell.ByFaces returned None — skip")
            continue
        eb      = _Shell.ExternalBoundary(s)
        ib_list = _Shell.InternalBoundaries(s)
        cf = Face.ByWires(eb, ib_list) if ib_list else Face.ByWire(eb)
        cf = Topology.RemoveCollinearEdges(cf) if cf else None
        if cf:
            floor_face_clean[fi] = cf
            print(f"  {FLOOR_NAMES[fi]}: clean face OK")
        else:
            print(f"  {FLOOR_NAMES[fi]}: face reconstruction returned None — skip")
    except Exception as ex:
        print(f"  {FLOOR_NAMES[fi]}: {ex} — skip")

for fi, lv in enumerate(FLOOR_LEVELS):
    plan_face = floor_face_clean.get(fi)
    if plan_face is None:
        print(f"\n{FLOOR_NAMES[fi]}: no clean face — isovist skipped (see VGA heatmap, section 19)")
        continue

    tris = extract_triangles(floor_faces[lv])
    uvp  = np.arange(UMIN, UMAX + ISO_STEP, ISO_STEP)
    vvp  = np.arange(VMIN, VMAX + ISO_STEP, ISO_STEP)
    UUv, VVv = np.meshgrid(uvp, vvp)
    vp_inside = np.column_stack([UUv.ravel(), VVv.ravel()])
    vp_inside = vp_inside[points_inside(tris, vp_inside)]
    print(f"\n{FLOOR_NAMES[fi]}: {len(vp_inside)} isovist viewpoints")

    iso_verts_list = [Vertex.ByCoordinates(float(u), float(v), 0.0) for u, v in vp_inside]
    floor_gverts   = [v for v in gverts if int(round(Vertex.Z(v) / FLOOR_HEIGHT)) == fi]

    valid_isovists, vis_counts = [], []
    for iv in iso_verts_list:
        try:
            iso = Face.Isovist(plan_face, iv)
            if iso is None:
                continue
            # Isovist may return a Cluster; extract the inner face
            iso_inner = Topology.Faces(iso)
            if not iso_inner:
                continue
            iso_face = iso_inner[0]
            count = sum(1 for gv in floor_gverts if Vertex.IsInternal2D(gv, iso_face))
            valid_isovists.append(iso_face)
            vis_counts.append(count)
        except Exception:
            pass

    if not valid_isovists:
        print(f"  No valid isovists — try a cleaner floor geometry")
        continue

    print(f"  Valid: {len(valid_isovists)} / {len(iso_verts_list)}")
    if vis_counts:
        print(f"  Visible cells — min: {min(vis_counts)}, max: {max(vis_counts)}, "
              f"mean: {sum(vis_counts)/len(vis_counts):.1f}")

    fig_iso = Topology.Show(
        [plan_face] + valid_isovists,
        faceOpacity=0.35, showEdges=False, showVertices=False,
        camera=[0, 0, 6], backgroundColor="black",
        width=1600, height=500,
        showFigure=False, renderer=renderer)
    if fig_iso:
        fig_iso.update_layout(title=dict(
            text=f"Isovist Analysis — {FLOOR_NAMES[fi]}",
            font=dict(color="white")))
        fig_iso.show(renderer=renderer)
        save_fig(fig_iso, f"11_isovists_{FLOOR_NAMES[fi].replace(' ', '_')}.png")

## 21. Building-wide Summary

All values come from the single connected building graph, so they describe the *whole* Unité d'Habitation section rather than any individual floor.

In [ ]:
def per_floor_mean(values):
    fl = np.array([int(round(Vertex.Z(v) / FLOOR_HEIGHT)) for v in gverts])
    vals = np.array(values, dtype=float)
    return [round(float(vals[fl == i].mean()), 4) for i in range(len(FLOOR_LEVELS))]

print(f"{'Metric':<24}{'min':>10}{'max':>10}{'mean':>10}   per-floor mean")
for name, vals in [("Degree centrality", degree_values),
                   ("Closeness centrality", closeness_values),
                   ("Betweenness centrality", betweenness_values)]:
    a = np.array(vals, dtype=float)
    print(f"{name:<24}{a.min():>10.4f}{a.max():>10.4f}{a.mean():>10.4f}   {per_floor_mean(vals)}")
print()
print(f"Nodes: {len(gverts)} | Edges: {len(gedges)} | Density: {Graph.Density(building_graph):.5f} "
      f"| Communities: {len(set(community_values))} | Stair edges: {len(stair_node_pairs)}")

In [ ]:
import numpy as np
arr = np.array(degree_values, dtype=float)
print("n =", len(arr))
print("min/max:", arr.min(), arr.max())
print("percentiles 50/90/99/100:", np.percentile(arr,[50,90,99,100]))
# how many distinct values and their counts
u,c = np.unique(np.round(arr,6), return_counts=True)
print("distinct:", len(u))
print("top values:", list(zip(u[-6:], c[-6:])))
print("nan?", np.isnan(arr).any())
# where a typical (median) value lands on a 0..1 linear scale
mn,mx = arr.min(), arr.max()
print("median maps to:", (np.median(arr)-mn)/(mx-mn))


## 22. Export analysis summary (notes + metadata)

Collect every metric computed above into a single human-readable **`analysis_summary.md`**
(notes) and a machine-readable **`analysis_metadata.json`**, both written to `ASSETS_DIR`.
Re-run this cell after a full run to refresh the report with the latest numbers.

In [ ]:
import json as _json
from datetime import datetime

def _stats(arr):
    a = np.asarray(arr, dtype=float)
    if a.size == 0:
        return dict(n=0, min=None, max=None, mean=None, std=None)
    return dict(n=int(a.size), min=float(a.min()), max=float(a.max()),
                mean=float(a.mean()), std=float(a.std()))

def _per_floor_mean(values):
    fl = np.array([int(round(Vertex.Z(v) / FLOOR_HEIGHT)) for v in gverts])
    vals = np.array(values, dtype=float)
    return [round(float(vals[fl == i].mean()), 6) if np.any(fl == i) else None
            for i in range(len(FLOOR_LEVELS))]

G = globals()
meta = {}
meta["generated"]   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
meta["notebook"]    = "NB_Marsella_Apartments_Section.ipynb"
meta["description"] = "Spatial Intelligence analysis of a 3-apartment section of the Unite d'Habitation (Marseille)."

# --- Parameters ---------------------------------------------------------------
meta["parameters"] = {
    "OBJ_PATH": OBJ_PATH,
    "GRID_SIZE": GRID_SIZE,
    "FLOOR_HEIGHT": FLOOR_HEIGHT,
    "FLOOR_LEVELS": FLOOR_LEVELS,
    "VGA_GRID_SIZE": G.get("VGA_GRID_SIZE"),
    "VIS_SAMPLES": G.get("VIS_SAMPLES"),
    "ISO_STEP": G.get("ISO_STEP"),
    "n_stair_locations": len(STAIR_LOCATIONS),
}

# --- Geometry -----------------------------------------------------------------
meta["geometry"] = {
    "plan_bbox": {"x_min": float(UMIN), "x_max": float(UMAX),
                  "y_min": float(VMIN), "y_max": float(VMAX),
                  "width": float(UMAX - UMIN), "height": float(VMAX - VMIN)},
    "faces_per_floor": {FLOOR_NAMES[i]: len(floor_faces[lv]) for i, lv in enumerate(FLOOR_LEVELS)},
    "navigable_nodes_per_floor": {FLOOR_NAMES[i]: int(len(floor_valid[lv])) for i, lv in enumerate(FLOOR_LEVELS)},
}

# --- Building graph -----------------------------------------------------------
meta["graph"] = {
    "nodes": len(gverts),
    "edges": len(gedges),
    "density": float(Graph.Density(building_graph)),
    "stair_edges": len(stair_node_pairs),
}

# --- Minimum Spanning Tree ----------------------------------------------------
if "mst" in G:
    meta["mst"] = {
        "vertices": len(mst_verts),
        "edges": len(mst_edges),
        "density": float(Graph.Density(mst)),
    }

# --- Shortest path ------------------------------------------------------------
if G.get("path") is not None:
    meta["shortest_path"] = {
        "length": float(path_len),
        "n_nodes": len(Topology.Vertices(path)),
        "simplified_length": float(simp_len) if "simp_len" in G else None,
        "simplified_waypoints": len(key_verts) if G.get("key_verts") else None,
    }

# --- Centralities -------------------------------------------------------------
meta["centrality"] = {}
for name, key in [("degree_values", "degree"),
                  ("closeness_values", "closeness"),
                  ("betweenness_values", "betweenness")]:
    if name in G:
        s = _stats(G[name])
        s["per_floor_mean"] = _per_floor_mean(G[name])
        meta["centrality"][key] = s

# Top hubs by degree
if "degree_values" in G:
    n_nodes = len(degree_values)
    top = sorted(zip(gverts, degree_values), key=lambda x: x[1], reverse=True)[:5]
    meta["centrality"]["top5_degree_hubs"] = [
        {"floor": int(round(Vertex.Z(v) / FLOOR_HEIGHT)) + 1,
         "x": round(Vertex.X(v), 2), "y": round(Vertex.Y(v), 2),
         "score": round(float(sc), 6), "connections": round(float(sc) * (n_nodes - 1))}
        for v, sc in top]

# --- Communities --------------------------------------------------------------
if "community_values" in G:
    from collections import Counter as _Counter
    cc = _Counter(community_values)
    cell_area = GRID_SIZE ** 2
    meta["communities"] = {
        "count": len(set(community_values)),
        "total_cells": len(community_values),
        "total_area_m2": round(len(community_values) * cell_area, 3),
        "sizes": {str(cid): {"cells": int(cnt), "area_m2": round(cnt * cell_area, 3)}
                  for cid, cnt in sorted(cc.items())},
    }

# --- Visibility (VGA) ---------------------------------------------------------
if "per_floor_vga" in G:
    meta["visibility_vga"] = {}
    for i, (xs, ys, deg) in enumerate(per_floor_vga):
        d = np.asarray(deg, dtype=float)
        meta["visibility_vga"][FLOOR_NAMES[i]] = {
            "viewpoints": int(d.size),
            "min": int(d.min()) if d.size else None,
            "max": int(d.max()) if d.size else None,
            "mean": round(float(d.mean()), 3) if d.size else None,
        }

# --- Write JSON ---------------------------------------------------------------
json_path = os.path.join(ASSETS_DIR, "analysis_metadata.json")
with open(json_path, "w", encoding="utf-8") as f:
    _json.dump(meta, f, ensure_ascii=False, indent=2)
print(f"Saved: {json_path}")

# --- Build Markdown notes -----------------------------------------------------
L = []
L.append(f"# Analysis Summary — Marsella 3-Apartment Section\n")
L.append(f"_Generated: {meta['generated']}_\n")
L.append(meta["description"] + "\n")

L.append("## Parameters\n")
L.append("| Parameter | Value |")
L.append("|---|---|")
for k, v in meta["parameters"].items():
    L.append(f"| `{k}` | {v} |")
L.append("")

g = meta["geometry"]["plan_bbox"]
L.append("## Geometry\n")
L.append(f"- **Plan bounding box:** X [{g['x_min']:.2f}, {g['x_max']:.2f}], "
         f"Y [{g['y_min']:.2f}, {g['y_max']:.2f}]  ({g['width']:.2f} × {g['height']:.2f} units)\n")
L.append("| Floor | Faces | Navigable nodes |")
L.append("|---|---|---|")
for i, lv in enumerate(FLOOR_LEVELS):
    fn = FLOOR_NAMES[i]
    L.append(f"| {fn} (Z={lv}) | {meta['geometry']['faces_per_floor'][fn]} | "
             f"{meta['geometry']['navigable_nodes_per_floor'][fn]} |")
L.append("")

gr = meta["graph"]
L.append("## Building graph\n")
L.append(f"- Nodes: **{gr['nodes']}**  |  Edges: **{gr['edges']}**  |  "
         f"Density: **{gr['density']:.5f}**  |  Stair edges: **{gr['stair_edges']}**\n")

if "mst" in meta:
    m = meta["mst"]
    L.append("## Minimum Spanning Tree\n")
    L.append(f"- Vertices: {m['vertices']}  |  Edges: {m['edges']}  |  Density: {m['density']:.5f}\n")

if "shortest_path" in meta:
    sp = meta["shortest_path"]
    L.append("## Cross-floor shortest path\n")
    L.append(f"- Length: **{sp['length']:.1f}**  |  Nodes: {sp['n_nodes']}")
    if sp.get("simplified_length") is not None:
        L.append(f"  |  Simplified: {sp['simplified_length']:.1f} "
                 f"({sp['simplified_waypoints']} waypoints)")
    L.append("")

if meta.get("centrality"):
    L.append("## Centrality metrics\n")
    L.append("| Metric | min | max | mean | std | per-floor mean |")
    L.append("|---|---|---|---|---|---|")
    for key in ("degree", "closeness", "betweenness"):
        s = meta["centrality"].get(key)
        if s:
            L.append(f"| {key.capitalize()} | {s['min']:.4f} | {s['max']:.4f} | "
                     f"{s['mean']:.4f} | {s['std']:.4f} | {s['per_floor_mean']} |")
    L.append("")
    if "top5_degree_hubs" in meta["centrality"]:
        L.append("**Top 5 degree hubs:**\n")
        for h in meta["centrality"]["top5_degree_hubs"]:
            L.append(f"- Floor {h['floor']} ({h['x']}, {h['y']}) → "
                     f"{h['score']:.4f} ({h['connections']} connections)")
        L.append("")

if "communities" in meta:
    cm = meta["communities"]
    L.append("## Community detection\n")
    L.append(f"- Communities: **{cm['count']}**  |  Total cells: {cm['total_cells']}  |  "
             f"Total area: ~{cm['total_area_m2']} m²\n")
    L.append("| Community | Cells | Area m² |")
    L.append("|---|---|---|")
    for cid, d in cm["sizes"].items():
        L.append(f"| {cid} | {d['cells']} | {d['area_m2']} |")
    L.append("")

if "visibility_vga" in meta:
    L.append("## Visibility Graph Analysis (per floor)\n")
    L.append("| Floor | Viewpoints | min | max | mean |")
    L.append("|---|---|---|---|---|")
    for fn, d in meta["visibility_vga"].items():
        L.append(f"| {fn} | {d['viewpoints']} | {d['min']} | {d['max']} | {d['mean']} |")
    L.append("")

L.append("---")
L.append("_Generated automatically by section 22 of the notebook. "
         "Community colours/partition are stochastic and may differ between runs._")

md_path = os.path.join(ASSETS_DIR, "analysis_summary.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(L))
print(f"Saved: {md_path}")
print("\n" + "\n".join(L[:18]))

## 23. Git commit & push

Commits the notebook, updated assets and summary files to the repository.
Edit `COMMIT_MSG` before running if you want a custom message.

In [ ]:
import subprocess, os
from datetime import datetime

REPO_ROOT = os.path.normpath(os.path.join(ASSETS_DIR, "..", "..", "..", ".."))
NB_PATH   = os.path.abspath(__vsc_ipynb_file__) if "__vsc_ipynb_file__" in dir() else             os.path.normpath(os.path.join(REPO_ROOT,
                "Final_Project", "Notebooks", "Spatial Intelligence",
                "NB_Marsella_Apartments_Section.ipynb"))

COMMIT_MSG = (f"Update apartment-section outputs — {datetime.now().strftime('%Y-%m-%d %H:%M')}")

def git(args, cwd=REPO_ROOT):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    return r.returncode

print(f"Repo root : {REPO_ROOT}")
print(f"Notebook  : {NB_PATH}")
print(f"Assets    : {ASSETS_DIR}")
print()

# Stage notebook + assets folder
git(["add", NB_PATH])
git(["add", ASSETS_DIR])

# Show what will be committed
git(["status", "--short"])
print()

rc = git(["commit", "-m", COMMIT_MSG])
if rc == 0:
    print("\nPushing...")
    git(["push", "origin", "main"])
else:
    print("Nothing to commit or commit failed — see status above.")